In [1]:
import torch
import torch.nn

In [ ]:
# <SOS> - 9 idx
# <EOS> - 10 idx

def BeamSearchDecoder(decoder, encoder_outs, hidden, beam_width=3, max_len=20, alpha=0.6):
  beam = [{"tokens": [9], "hidden": hidden, "score": 0.0}]

  step = 0

  while step < max_len:
    if all(hyp["tokens"][-1] == 10 for hyp in beam): break
    step+=1

    candidates = []

    for hyp in beam:
      if hyp["tokens"][-1] == 10:
        candidates.append(hyp)
        continue

      logits, hidden2 = decoder.forward_step(torch.tensor([hyp["tokens"][-1]]).to(hidden.device), hyp["hidden"], encoder_outs)

      log_probs = torch.log_softmax(logits, dim=-1)

      for token_id in range(log_probs.size(-1)):
        new_score = (hyp["score"]+log_probs[0, token_id].item())

        candidates.append({"tokens": hyp["tokens"] + [token_id], "hidden": hidden2, "score": new_score})


    scores = torch.tensor([x["score"] for x in candidates])

    top_scores, top_ids = torch.topk(scores, k=beam_width)
    beam = [candidates[i] for i in top_ids]


  return beam

